Francisco Benita, 2026

In [1]:
boundaries_dict = {
"Jakarta": "https://www.dropbox.com/scl/fi/e407ghh550cve039dpg2w/Jakarta_boundary.gpkg?rlkey=a75ym7fbgh2v0bxs321zr6g7z&st=qgvxfn0s&dl=1",
"Manila": "https://www.dropbox.com/scl/fi/l08g71niyf49tag16ezda/Manila_boundary.gpkg?rlkey=yrsjkklyp2cf3x1t0jjswe7i7&st=11b7ughn&dl=1",
"Ho Chi Minh City": "https://www.dropbox.com/scl/fi/zeot0bz2cwznoaqre2e0f/Ho_Chi_Minh_City_boundary.gpkg?rlkey=pa9pd5omwvd02cqcluj8ivxza&st=73lf8tlu&dl=1",
"Phnom Penh": "https://www.dropbox.com/scl/fi/lu6b5r7gseg1dlxdvy6z6/Phnom_Penh_boundary.gpkg?rlkey=e9o9k04rjr1xebtduqlrsu7xj&st=8i5k701j&dl=1"
}
print(boundaries_dict)

{'Jakarta': 'https://www.dropbox.com/scl/fi/e407ghh550cve039dpg2w/Jakarta_boundary.gpkg?rlkey=a75ym7fbgh2v0bxs321zr6g7z&st=qgvxfn0s&dl=1', 'Manila': 'https://www.dropbox.com/scl/fi/l08g71niyf49tag16ezda/Manila_boundary.gpkg?rlkey=yrsjkklyp2cf3x1t0jjswe7i7&st=11b7ughn&dl=1', 'Ho Chi Minh City': 'https://www.dropbox.com/scl/fi/zeot0bz2cwznoaqre2e0f/Ho_Chi_Minh_City_boundary.gpkg?rlkey=pa9pd5omwvd02cqcluj8ivxza&st=73lf8tlu&dl=1', 'Phnom Penh': 'https://www.dropbox.com/scl/fi/lu6b5r7gseg1dlxdvy6z6/Phnom_Penh_boundary.gpkg?rlkey=e9o9k04rjr1xebtduqlrsu7xj&st=8i5k701j&dl=1'}


# OpenStreetMaps - Food amenities

In [2]:
food_tags = {'amenity': ["bar", "cafe", "fast_food", "food_court", "restaurant"]}

In [3]:
!pip install osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.4 MB/s eta 0:00:00


In [4]:
import osmnx as ox
import geopandas as gpd
import requests
import os

First, let's define a function to download the boundary file, extract the amenities using `osmnx`, and then save the amenities to a GeoPackage file for each city.

In [5]:
def extract_and_save_amenities(city_name, boundary_url, amenity_tags, amenity_type):
    print(f"Processing {amenity_type} amenities for {city_name}...")

    # Define local path for boundary file
    boundary_gpkg_path = f"./{city_name.replace(' ', '_')}_boundary.gpkg"

    # Download boundary file if not already present
    if not os.path.exists(boundary_gpkg_path):
        try:
            print(f"Downloading boundary for {city_name} from {boundary_url}...")
            response = requests.get(boundary_url, stream=True)
            response.raise_for_status() # Raise an exception for HTTP errors
            with open(boundary_gpkg_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"Downloaded {city_name} boundary to {boundary_gpkg_path}")
        except requests.exceptions.RequestException as e:
            print(f"Error downloading boundary for {city_name}: {e}")
            return
    else:
        print(f"Boundary for {city_name} already exists at {boundary_gpkg_path}")

    # Load the boundary GeoPackage file
    try:
        city_boundary = gpd.read_file(boundary_gpkg_path)
        # Ensure the boundary is a single polygon or multipolygon
        if not city_boundary.empty:
            # Take the largest polygon if there are multiple, or combine if necessary
            if len(city_boundary) > 1:
                city_boundary_geom = city_boundary.geometry.unary_union
            else:
                city_boundary_geom = city_boundary.geometry.iloc[0]
        else:
            print(f"No valid geometry found in boundary file for {city_name}.")
            return

    except Exception as e:
        print(f"Error reading boundary file for {city_name}: {e}")
        return

    # Extract amenities using osmnx
    print(f"Extracting {amenity_type} amenities for {city_name}...")
    try:
        # Ensure we pass a shapely geometry object to features_from_polygon
        amenities = ox.features_from_polygon(city_boundary_geom, amenity_tags)

        # Filter to keep only point geometries
        amenities = amenities[amenities.geometry.geom_type.isin(['Point', 'MultiPoint'])]

        print(f"Found {len(amenities)} point {amenity_type} amenities in {city_name}.")

        # Define output path for amenities GeoPackage
        output_gpkg_path = f"./{city_name.replace(' ', '_')}_{amenity_type}_amenities.gpkg"

        # Save amenities to a GeoPackage file
        if not amenities.empty:
            # Drop problematic columns if they exist
            columns_to_drop = ['FIXME', 'Status']
            for col in columns_to_drop:
                if col in amenities.columns:
                    amenities = amenities.drop(columns=[col])

            amenities.to_file(output_gpkg_path, driver='GPKG')
            print(f"Saved {amenity_type} amenities for {city_name} to {output_gpkg_path}")
        else:
            print(f"No {amenity_type} amenities to save for {city_name}.")

    except Exception as e:
        print(f"Error extracting or saving {amenity_type} amenities for {city_name}: {e}")


In [6]:
healthcare_tags = {'amenity': ["clinic", "doctors", "pharmacy", "dentist", "hospital", "veterinary"]}
education_tags = {'amenity': ["school", "college", "university", "kindergarten", "library", "training"]}
leisure_tags = {'amenity': ["cinema", "theatre", "arts_centre", "museum", "community_centre", "place_of_worship", "park", "playground", "sports_centre", "fitness_centre", "stadium", "swimming_pool"]}

# Combine all amenity types and their tags
amenity_categories = {
    "food": food_tags,
    "healthcare": healthcare_tags,
    "education": education_tags,
    "leisure": leisure_tags
}

# Loop through each city and each amenity category to extract and save data
for city, url in boundaries_dict.items():
    for amenity_type, tags in amenity_categories.items():
        extract_and_save_amenities(city, url, tags, amenity_type)

print("Processing complete for all cities and all specified amenity types.")

Processing food amenities for Jakarta...
Downloaded Jakarta boundary to ./Jakarta_boundary.gpkg
Extracting food amenities for Jakarta...
Found 2394 point food amenities in Jakarta.
Saved food amenities for Jakarta to ./Jakarta_food_amenities.gpkg
Processing healthcare amenities for Jakarta...
Boundary for Jakarta already exists at ./Jakarta_boundary.gpkg
Extracting healthcare amenities for Jakarta...
Found 1066 point healthcare amenities in Jakarta.
Saved healthcare amenities for Jakarta to ./Jakarta_healthcare_amenities.gpkg
Processing education amenities for Jakarta...
Boundary for Jakarta already exists at ./Jakarta_boundary.gpkg
Extracting education amenities for Jakarta...
Found 616 point education amenities in Jakarta.
Saved education amenities for Jakarta to ./Jakarta_education_amenities.gpkg
Processing leisure amenities for Jakarta...
Boundary for Jakarta already exists at ./Jakarta_boundary.gpkg
Extracting leisure amenities for Jakarta...
Found 228 point leisure amenities in J

In [7]:
!pip freeze > colab_environment_snapshot.txt